In [1]:
import pandas as pd
import numpy as np

In [4]:
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [17]:
import os

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_0a3c93a5077f0a7c586a717e04524399'
from kaggle.api.kaggle_api_extended import KaggleApi


In [24]:
api = KaggleApi()
api.authenticate()

dataset_id = 'razanihababdellatif/food-delivery-orders-and-eta-logistics-dataset'
api.dataset_download_files(dataset_id, path='.', unzip=True)
csv_file = [f for f in os.listdir() if f.endswith('.csv')][0]
df = pd.read_csv(csv_file)


Dataset URL: https://www.kaggle.com/datasets/razanihababdellatif/food-delivery-orders-and-eta-logistics-dataset


In [25]:
df['eta_error_min'] = df['actual_delivery_time_minutes'] - df['estimated_delivery_time_minutes']
df['tiempo_transito_min'] = df['actual_delivery_time_minutes'] - df['restaurant_preparation_time_minutes']
df['pct_tiempo_cocina'] = (df['restaurant_preparation_time_minutes'] / df['actual_delivery_time_minutes']) * 100
df['status_entrega'] = df['late_delivery'].map({0.0: 'A tiempo', 1.0: 'Tardío'})


In [26]:
matriz_trafico_cocina = df.pivot_table(
    index='traffic_level',
    columns='status_entrega',
    values='restaurant_preparation_time_minutes',
    aggfunc='mean'
).reindex(['Low', 'Medium', 'High'])

matriz_trafico_cocina.columns.name = 'Estado de Entrega'
matriz_trafico_cocina.index.name = 'Nivel de Tráfico'

print("=== Tiempo Promedio en Cocina (Minutos) ===")
matriz_trafico_cocina

=== Tiempo Promedio en Cocina (Minutos) ===


Estado de Entrega,A tiempo,Tardío
Nivel de Tráfico,,
Low,14.13,18.40
Medium,14.16,18.03
High,14.06,17.63


In [27]:
df['eta_error_min'] = df['actual_delivery_time_minutes'] - df['estimated_delivery_time_minutes']

# 2. Identificamos la columna exacta de clima (weather o weather_conditions)
col_clima = 'weather' if 'weather' in df.columns else 'weather_conditions'

# 3. Generamos la tabla dinámica
matriz_eta_error = df.pivot_table(
    index=col_clima,
    columns='traffic_level',
    values='eta_error_min',
    aggfunc='mean'
)

# Reordenamos columnas de tráfico si existen
columnas_trafico = [c for c in ['Low', 'Medium', 'High'] if c in matriz_eta_error.columns]
matriz_eta_error = matriz_eta_error[columnas_trafico]

print(f"=== Minutos Promedio de Desvío sobre el ETA (Usando columna '{col_clima}') ===")
matriz_eta_error

=== Minutos Promedio de Desvío sobre el ETA (Usando columna 'weather') ===


traffic_level,Low,Medium,High
weather,,,
Clear,2.54,2.60,2.48
Cloudy,3.94,4.26,4.68
Rain,7.16,8.35,10.07
Storm,12.54,15.95,18.59


In [28]:
desempeno_ciudad = df.groupby('city').agg(
    total_pedidos=('order_id', 'count'),
    tiempo_cocina_prom=('restaurant_preparation_time_minutes', 'mean'),
    tiempo_entrega_prom=('actual_delivery_time_minutes', 'mean'),
    tasa_retraso=('late_delivery', lambda x: (x.mean() * 100))
).rename(columns={'tasa_retraso': 'tasa_retraso_%'})

print("=== Desempeño Operativo por Ciudad ===")
desempeno_ciudad.sort_values(by='tasa_retraso_%', ascending=False)

=== Desempeño Operativo por Ciudad ===


,total_pedidos,tiempo_cocina_prom,tiempo_entrega_prom,tasa_retraso_%
city,,,,
City_B,14885,16.08,36.01,38.19
City_A,19888,15.25,34.99,37.59
City_D,5049,16.27,35.47,34.73
City_C,10178,15.00,34.06,33.12


In [29]:
output_file = 'power_bi_delivery_dataset.csv'
df.to_csv(output_file, index=False)

print(f"💾 Archivo '{output_file}' generado exitosamente.")

💾 Archivo 'power_bi_delivery_dataset.csv' generado exitosamente.
